# Phase 4: Customer Segmentation

In this phase, we segment customers based on behavioral and monetary characteristics.

Objectives:
- Identify distinct customer groups
- Compare RFM-based segmentation and ML clustering
- Generate actionable business insights

Two approaches are used:
1. RFM Quantile-Based Segmentation
2. KMeans Clustering

In [ ]:
# ============================================
# 1. Imports
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)

In [ ]:
# ============================================
# Load Customer-Level Dataset
# ============================================

customer_df = pd.read_csv("../data/processed/customer_dataset.csv")

print("Shape:", customer_df.shape)
customer_df.head()

## 1. RFM Quantile-Based Segmentation

We assign customers into quartiles based on:
- Recency (lower is better)
- Frequency (higher is better)
- Monetary (higher is better)

In [ ]:
# ============================================
# RFM Scoring (Quartiles)
# ============================================

rfm = customer_df[['customer_id', 'recency_days', 'frequency', 'monetary']].copy()

rfm['R_score'] = pd.qcut(rfm['recency_days'], 4, labels=[4,3,2,1]).astype(int)
rfm['F_score'] = rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'),4,labels=[1,2,3,4]
).astype(int)
rfm['M_score'] = rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'),4,labels=[1,2,3,4]
).astype(int)

rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

rfm.head()

In [ ]:
# ============================================
# Segment Classification(converting rfm scores to segments)
# ============================================

def segment_customers(row):            # each row = each customer
    if row['RFM_score'] >= 9:
        return "High Value"
    elif row['RFM_score'] >= 6:
        return "Mid Value"
    else:
        return "Low Value"

rfm['segment'] = rfm.apply(segment_customers, axis=1) #Apply the function to every customer.

rfm['segment'].value_counts()

In [ ]:
plt.figure()
rfm['segment'].value_counts().plot(kind='bar')
plt.title("RFM Segment Distribution")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")
plt.show()

## 2. KMeans Clustering

We apply unsupervised machine learning to identify
natural customer groupings based on behavioral metrics.

In [ ]:
# ============================================
# Feature Selection
# ============================================

features = customer_df[['recency_days', 'frequency', 'monetary',
                        'avg_order_value', 'avg_review_score']].copy()

In [ ]:
# ============================================
# Handle Missing Values
# ============================================

# Replace review_score NaN with median (neutral assumption)
features['avg_review_score'] = features['avg_review_score'].fillna(
    features['avg_review_score'].median()
)

# Replace any other remaining NaN with column median
features = features.fillna(features.median())

In [ ]:
# ============================================
# Feature Scaling
# ============================================

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 11), inertia)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
# ============================================
# Apply KMeans (Example with 3 clusters)
# ============================================

kmeans = KMeans(n_clusters=3, random_state=42)
customer_df['cluster'] = kmeans.fit_predict(scaled_features)

customer_df['cluster'].value_counts()

## 3. Cluster Profiling

We analyze cluster characteristics by comparing
average metrics across clusters.

In [ ]:
cluster_profile = (
    customer_df
    .groupby('cluster')
    .agg({
        'recency_days': 'mean',
        'frequency': 'mean',
        'monetary': 'mean',
        'avg_order_value': 'mean',
        'avg_review_score': 'mean'
    })
    .round(2)   # converta decimal places upto only 2 digits after .
)

cluster_profile

In [ ]:
cluster_profile.plot(kind='bar', figsize=(12,6))
plt.title("Cluster Profiling")
plt.ylabel("Average Value")
plt.show()

## Segmentation Insights

RFM Segmentation:
- High Value customers contribute significant revenue.
- Low Value customers represent potential retention risk.

KMeans Clustering:
- Cluster 0: High-value engaged customers.
- Cluster 1: Mid-tier customers.
- Cluster 2: Low-frequency / low-monetary customers.

Business Actions:
- Target high-value customers with loyalty programs.
- Re-engage mid-tier customers with personalized offers.
- Design retention campaigns for low-value segment.